In [ ]:
import glob
import json
import os
import uuid

import cv2
from dotenv import load_dotenv
import numpy as np
from PIL import Image

import torch

import matplotlib.pyplot as plt

from koger_detection.obj_det.predictors import Predictor
from koger_detection.obj_det.mydatasets import ImageDataset
from koger_detection.obj_det.engine import collate_fn, worker_init_fn

/home/koger/environments/orthocounting/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Assumes we have a local .env file that stores things like ROOT
load_dotenv()
model_name = "03-24-2025-11-46-27"
images_folder = "/mnt/h/Pronghorn Vertical Imagery/2024/PR632"

# Where outputs will be saved
output_folder = os.path.join("/home/koger/pronghorn-processing", model_name)
os.makedirs(output_folder, exist_ok=True)

research_project = "pronghorn-survey"

In [3]:


model_path = os.environ.get("MODEL_PATH")
model_folder = os.path.join(model_path, research_project, "runs", model_name)
model_cfg_file = os.path.join(model_folder, "cfg.json")
model_weights_file = os.path.join(model_folder, "final_model.pth")

image_files = sorted(glob.glob(os.path.join(images_folder, f"*.[jJ][pP][gG]")))
print(f"{len(image_files)} files found.")

14579 files found.


In [ ]:
with open(model_cfg_file) as f:
    cfg = json.load(f)
    model_cfg = cfg['model']
# Should be the resolution of the raw images being processed
model_cfg['fixed_size'] = [8256, 5504]
model_cfg['model_weights_pth'] = model_weights_file

In [ ]:
rgb = True
image_dataset = ImageDataset(image_files, rgb=rgb)
dataloader = torch.utils.data.DataLoader(
            image_dataset, batch_size=1, shuffle=False, 
            num_workers=4)

predictor = Predictor(model_cfg, invert_color_channel=False)

In [ ]:
for ind, image in enumerate(dataloader):
    if ind % 300 == 0:
        print(ind)

    image_data = image['image'][0]
    if (image_data.shape[0] == 2) and (image_data.shape[1] == 2):
        # This isn't beautiful. When Dataset can't read image returns a 2x2x3 array.
        print("Skipping unreadable image.")
        continue
                                       
    res = predictor(image_data)
    
    boxes = res['boxes'].to('cpu').numpy().astype(np.uint32)
    scores = res['scores'].to('cpu').numpy()
    labels = res['labels'].to('cpu').detach().numpy()

    image_name = os.path.splitext(os.path.basename(image['filename'][0]))[0]

    np.save(os.path.join(output_folder, f"{image_name}_boxes.npy"), boxes)
    np.save(os.path.join(output_folder, f"{image_name}_scores.npy"), scores)
    np.save(os.path.join(output_folder, f"{image_name}_labels.npy"), labels)
